# NPB → MLB Player Matching

Match NPB players to MLB players, identify the first MLB season after NPB experience, classify player roles, and create the final transition datasets.

In [ ]:
from pathlib import Path
import pandas as pd

In [13]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

npb_batting = pd.read_csv(PROCESSED / "npb_batting.csv")
npb_pitching = pd.read_csv(PROCESSED / "npb_pitching.csv")

mlb_batting = pd.read_csv(PROCESSED / "mlb_batting.csv")
mlb_pitching = pd.read_csv(PROCESSED / "mlb_pitching.csv")
mlb_people = pd.read_csv(PROCESSED / "mlb_people.csv")

In [2]:
def split_npb_name(name):
    name = str(name).strip()

    if name == "Oh Seung-hwan":
        return pd.Series(
            {"LastName": "Oh", "FirstName": "Seung-hwan"}
        )

    last, first = name.split(",", 1)

    return pd.Series(
        {"LastName": last.strip(), "FirstName": first.strip()}
    )


npb_batting[["LastName", "FirstName"]] = npb_batting["Name"].apply(split_npb_name)
npb_pitching[["LastName", "FirstName"]] = npb_pitching["Name"].apply(split_npb_name)

npb_players = (
    pd.concat([
        npb_batting[["PlayerID", "FirstName", "LastName"]],
        npb_pitching[["PlayerID", "FirstName", "LastName"]]
    ])
    .drop_duplicates()
)

mlb_players = mlb_people[
    ["playerID", "nameFirst", "nameLast", "birthYear", "birthCountry"]
].rename(
    columns={"nameFirst": "FirstName", "nameLast": "LastName"}
)

In [3]:
name_matches = npb_players.merge(
    mlb_players,
    on=["FirstName", "LastName"],
    how="inner"
)

unique_name_matches = name_matches.groupby("PlayerID")["playerID"].nunique()
unique_npb_matches = unique_name_matches[unique_name_matches == 1].index

unambiguous_matches = name_matches[
    name_matches["PlayerID"].isin(unique_npb_matches)
].copy()

In [4]:
npb_seasons = (
    pd.concat([
        npb_batting[["PlayerID", "Season"]],
        npb_pitching[["PlayerID", "Season"]]
    ])
    .drop_duplicates()
    .groupby("PlayerID")["Season"]
    .apply(lambda x: sorted(x.tolist()))
    .to_dict()
)

mlb_seasons = (
    pd.concat([
        mlb_batting[["playerID", "yearID"]],
        mlb_pitching[["playerID", "yearID"]]
    ])
    .drop_duplicates()
    .groupby("playerID")["yearID"]
    .apply(lambda x: sorted(x.tolist()))
    .to_dict()
)

mlb_first_last = (
    pd.concat([
        mlb_batting[["playerID", "yearID"]],
        mlb_pitching[["playerID", "yearID"]]
    ])
    .groupby("playerID")["yearID"]
    .agg(MLB_First_Season="min", MLB_Last_Season="max")
)

In [5]:
transitions = []

for _, row in unambiguous_matches.iterrows():
    npb_years = npb_seasons.get(row["PlayerID"], [])
    mlb_years = mlb_seasons.get(row["playerID"], [])

    possible_mlb_transitions = [
        year
        for year in mlb_years
        if any(npb_year < year for npb_year in npb_years)
    ]

    if not possible_mlb_transitions:
        continue

    mlb_transition_season = min(possible_mlb_transitions)
    npb_transition_season = max(
        year for year in npb_years if year < mlb_transition_season
    )

    transitions.append({
        "PlayerID": row["PlayerID"],
        "MLB_playerID": row["playerID"],
        "FirstName": row["FirstName"],
        "LastName": row["LastName"],
        "NPB_Transition_Season": npb_transition_season,
        "MLB_Transition_Season": mlb_transition_season
    })

transitions = pd.DataFrame(transitions)

transitions["MLB_First_Season"] = transitions["MLB_playerID"].map(
    mlb_first_last["MLB_First_Season"]
)
transitions["MLB_Last_Season"] = transitions["MLB_playerID"].map(
    mlb_first_last["MLB_Last_Season"]
)

transitions["First_MLB_After_NPB"] = (
    transitions["MLB_Transition_Season"]
    == transitions["MLB_First_Season"]
)

npb_mlb_candidates = transitions[
    transitions["First_MLB_After_NPB"]
].copy()

npb_mlb_candidates["Years_Between"] = (
    npb_mlb_candidates["MLB_Transition_Season"]
    - npb_mlb_candidates["NPB_Transition_Season"]
)

In [6]:
npb_career_length = (
    pd.concat([
        npb_batting[["PlayerID", "Season"]],
        npb_pitching[["PlayerID", "Season"]]
    ])
    .drop_duplicates()
    .groupby("PlayerID")
    .size()
    .rename("NPB_Seasons")
)

mlb_career_length = (
    pd.concat([
        mlb_batting[["playerID", "yearID"]],
        mlb_pitching[["playerID", "yearID"]]
    ])
    .drop_duplicates()
    .groupby("playerID")
    .size()
    .rename("MLB_Seasons")
)

npb_mlb_candidates["NPB_Seasons"] = npb_mlb_candidates["PlayerID"].map(npb_career_length)
npb_mlb_candidates["MLB_Seasons"] = npb_mlb_candidates["MLB_playerID"].map(mlb_career_length)

npb_teams = (
    pd.concat([
        npb_batting[["PlayerID", "Season", "Team"]],
        npb_pitching[["PlayerID", "Season", "Team"]]
    ])
    .drop_duplicates(["PlayerID", "Season", "Team"])
    .groupby(["PlayerID", "Season"])["Team"]
    .agg(lambda x: "; ".join(x.astype(str).unique()))
    .rename("NPB_Team")
    .reset_index()
)

mlb_teams = (
    pd.concat([
        mlb_batting[["playerID", "yearID", "teamID"]],
        mlb_pitching[["playerID", "yearID", "teamID"]]
    ])
    .drop_duplicates(["playerID", "yearID", "teamID"])
    .groupby(["playerID", "yearID"])["teamID"]
    .agg(lambda x: "; ".join(x.astype(str).unique()))
    .rename("MLB_Team")
    .reset_index()
)

npb_mlb_candidates = (
    npb_mlb_candidates
    .merge(
        npb_teams,
        left_on=["PlayerID", "NPB_Transition_Season"],
        right_on=["PlayerID", "Season"],
        how="left"
    )
    .drop(columns="Season")
    .merge(
        mlb_teams,
        left_on=["MLB_playerID", "MLB_Transition_Season"],
        right_on=["playerID", "yearID"],
        how="left"
    )
    .drop(columns=["playerID", "yearID"])
)

In [7]:
birth_info = (
    mlb_people[
        ["playerID", "birthYear", "birthCountry"]
    ]
    .drop_duplicates("playerID")
)

npb_mlb_candidates = npb_mlb_candidates.merge(
    birth_info,
    left_on="MLB_playerID",
    right_on="playerID",
    how="left"
).drop(columns="playerID")

npb_mlb_candidates["birthYear"] = pd.to_numeric(
    npb_mlb_candidates["birthYear"],
    errors="coerce"
)

validated_candidates = npb_mlb_candidates[
    npb_mlb_candidates["birthYear"].isna()
    | (
        npb_mlb_candidates["birthYear"]
        <= npb_mlb_candidates["NPB_Transition_Season"]
    )
].copy()

In [8]:
transition_position = npb_batting[
    ["PlayerID", "Season", "Position"]
].rename(
    columns={"Season": "NPB_Transition_Season"}
)

validated_candidates = validated_candidates.merge(
    transition_position,
    on=["PlayerID", "NPB_Transition_Season"],
    how="left"
)

validated_candidates["NPB_Role"] = "Hitter"
validated_candidates.loc[
    validated_candidates["Position"] == "P",
    "NPB_Role"
] = "Pitcher"

ohtani_mask = (
    validated_candidates["FirstName"].eq("Shohei")
    & validated_candidates["LastName"].eq("Ohtani")
)

validated_candidates.loc[ohtani_mask, "NPB_Role"] = "Two-way"

validated_candidates["NPB_Role"].value_counts()

NPB_Role
Pitcher    60
Hitter     32
Two-way     1
Name: count, dtype: int64

In [9]:
hitter_candidates = validated_candidates[
    validated_candidates["NPB_Role"] == "Hitter"
][
    [
        "PlayerID",
        "FirstName",
        "LastName",
        "NPB_Transition_Season",
        "MLB_Transition_Season",
        "MLB_playerID"
    ]
].copy()

npb_hitter_features = npb_batting.merge(
    hitter_candidates[["PlayerID", "NPB_Transition_Season"]],
    left_on=["PlayerID", "Season"],
    right_on=["PlayerID", "NPB_Transition_Season"],
    how="inner"
)[
    [
        "PlayerID", "Season", "PA", "AB", "H", "HR", "RBI", "SB",
        "BB", "K", "BA", "OBP", "SLG", "OPS"
    ]
].rename(columns={
    "Season": "NPB_Season",
    "PA": "NPB_PA",
    "AB": "NPB_AB",
    "H": "NPB_H",
    "HR": "NPB_HR",
    "RBI": "NPB_RBI",
    "SB": "NPB_SB",
    "BB": "NPB_BB",
    "K": "NPB_K",
    "BA": "NPB_BA",
    "OBP": "NPB_OBP",
    "SLG": "NPB_SLG",
    "OPS": "NPB_OPS"
})

mlb_hitter_features = mlb_batting.merge(
    hitter_candidates[["MLB_playerID", "MLB_Transition_Season"]],
    left_on=["playerID", "yearID"],
    right_on=["MLB_playerID", "MLB_Transition_Season"],
    how="inner"
)[
    [
        "MLB_playerID", "yearID", "G", "AB", "H", "HR", "RBI", "SB",
        "BB", "SO", "BA", "OBP", "SLG", "OPS"
    ]
].rename(columns={
    "yearID": "MLB_Season",
    "G": "MLB_G",
    "AB": "MLB_AB",
    "H": "MLB_H",
    "HR": "MLB_HR",
    "RBI": "MLB_RBI",
    "SB": "MLB_SB",
    "BB": "MLB_BB",
    "SO": "MLB_K",
    "BA": "MLB_BA",
    "OBP": "MLB_OBP",
    "SLG": "MLB_SLG",
    "OPS": "MLB_OPS"
})

npb_mlb_hitters = (
    hitter_candidates
    .merge(npb_hitter_features, on="PlayerID", how="inner")
    .merge(mlb_hitter_features, on="MLB_playerID", how="inner")
)

In [10]:
pitcher_candidates = validated_candidates[
    validated_candidates["NPB_Role"] == "Pitcher"
][
    [
        "PlayerID",
        "FirstName",
        "LastName",
        "NPB_Transition_Season",
        "MLB_Transition_Season",
        "MLB_playerID"
    ]
].copy()

npb_pitcher_features = npb_pitching.merge(
    pitcher_candidates[["PlayerID", "NPB_Transition_Season"]],
    left_on=["PlayerID", "Season"],
    right_on=["PlayerID", "NPB_Transition_Season"],
    how="inner"
)[
    [
        "PlayerID", "Season", "IP", "W", "L", "G", "GS", "S",
        "K", "BB", "HR", "ERA", "WHIP", "K/9", "BB/9", "HR/9", "K/BB"
    ]
].rename(columns={
    "Season": "NPB_Season",
    "IP": "NPB_IP",
    "W": "NPB_W",
    "L": "NPB_L",
    "G": "NPB_G",
    "GS": "NPB_GS",
    "S": "NPB_S",
    "K": "NPB_K",
    "BB": "NPB_BB",
    "HR": "NPB_HR",
    "ERA": "NPB_ERA",
    "WHIP": "NPB_WHIP",
    "K/9": "NPB_K9",
    "BB/9": "NPB_BB9",
    "HR/9": "NPB_HR9",
    "K/BB": "NPB_KBB"
})

mlb_pitcher_features = mlb_pitching.merge(
    pitcher_candidates[["MLB_playerID", "MLB_Transition_Season"]],
    left_on=["playerID", "yearID"],
    right_on=["MLB_playerID", "MLB_Transition_Season"],
    how="inner"
)[
    [
        "MLB_playerID", "yearID", "IP", "W", "L", "G", "GS", "SV",
        "SO", "BB", "HR", "ERA", "WHIP", "K/9", "BB/9", "HR/9", "K/BB"
    ]
].rename(columns={
    "yearID": "MLB_Season",
    "IP": "MLB_IP",
    "W": "MLB_W",
    "L": "MLB_L",
    "G": "MLB_G",
    "GS": "MLB_GS",
    "SV": "MLB_SV",
    "SO": "MLB_K",
    "BB": "MLB_BB",
    "HR": "MLB_HR",
    "ERA": "MLB_ERA",
    "WHIP": "MLB_WHIP",
    "K/9": "MLB_K9",
    "BB/9": "MLB_BB9",
    "HR/9": "MLB_HR9",
    "K/BB": "MLB_KBB"
})

npb_mlb_pitchers = (
    pitcher_candidates
    .merge(npb_pitcher_features, on="PlayerID", how="inner")
    .merge(mlb_pitcher_features, on="MLB_playerID", how="inner")
)

In [11]:
MIN_NPB_PA = 200
MIN_NPB_IP = 30

final_hitters = npb_mlb_hitters[
    npb_mlb_hitters["NPB_PA"] >= MIN_NPB_PA
].copy()

final_pitchers = npb_mlb_pitchers[
    npb_mlb_pitchers["NPB_IP"] >= MIN_NPB_IP
].copy()


(final_hitters.shape, final_pitchers.shape)

((26, 32), (50, 38))

In [12]:
final_hitters.to_csv(
    PROCESSED / "npb_mlb_hitters.csv",
    index=False
)

final_pitchers.to_csv(
    PROCESSED / "npb_mlb_pitchers.csv",
    index=False
)